# Dedekind DSL: Expert Tier

This notebook sketches an **expert-friendly DSL** for set operations using mathematical notation and terminology.

**Design philosophy**: Use mathematically precise names and operations for users familiar with set theory and formal logic.

## Scope & Prototype Status

This is a **prototype DSL shim** demonstrating desired ergonomics for mathematical users. The goal is to show:
- **Intensional** definitions: specify sets via predicates and comprehensions
- **Extensional** evaluation: compute to concrete results
- Mathematical notation: ∈, ⊆, ∪, ∩, ∖ (where notation is practical in Python)

Outputs may be approximate; correctness refinement is a follow-up item (issue #241).

In [ ]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing pyproject.toml")

repo_root = find_repo_root(Path.cwd())

try:
    import dedekind
except ModuleNotFoundError as err:
    raise ModuleNotFoundError(
        "dedekind is not installed in this notebook kernel. "
        f"From {repo_root}, run `python -m pip install -e .` "
        "or use `make integration-test` for the CI-style execution path."
    ) from err

print("expert-tier: dedekind import ok")

In [ ]:
class Ensemble:
    """Expert-tier set representation using mathematical terminology.
    
    'Ensemble' emphasizes that this is a formal mathematical object,
    not just a Python container. Supports both intensional and extensional
    representations.
    """

    def __init__(self, members=None, predicate=None, universe=None):
        # Extensional: explicit members
        self._members = list(members) if members is not None else None
        # Intensional: predicate on universe
        self._predicate = predicate
        self._universe = universe or set()

    @staticmethod
    def from_members(*items):
        """Create extensional ensemble from members: {a, b, c, ...}."""
        return Ensemble(members=items)

    @staticmethod
    def comprehension(universe, predicate):
        """Create intensional ensemble via set-builder: {x ∈ U | P(x)}."""
        return Ensemble(
            predicate=predicate,
            universe=universe
        )

    def restrict(self, predicate):
        """Restrict to elements satisfying a property: {x ∈ S | Q(x)}.
        
        Corresponds to beginner 'where' but with mathematical naming.
        """
        if self._members is not None:
            restricted = [v for v in self._members if predicate(v)]
            return Ensemble(members=restricted)
        else:
            # Compose predicates
            return Ensemble(
                predicate=lambda x: (self._predicate(x) if self._predicate else True) and predicate(x),
                universe=self._universe
            )

    def map_to(self, morphism):
        """Apply morphism (function) to each element: f(S) = {f(x) | x ∈ S}."""
        if self._members is not None:
            image = [morphism(v) for v in self._members]
            return Ensemble(members=image)
        return self  # Fallback for symbolic

    def union(self, other):
        """Ensemble union: A ∪ B."""
        if self._members is not None and other._members is not None:
            result = list(set(self._members) | set(other._members))
            return Ensemble(members=result)
        return self  # Fallback

    def intersection(self, other):
        """Ensemble intersection: A ∩ B."""
        if self._members is not None and other._members is not None:
            result = list(set(self._members) & set(other._members))
            return Ensemble(members=result)
        return self  # Fallback

    def difference(self, other):
        """Ensemble difference (relative complement): A ∖ B."""
        if self._members is not None and other._members is not None:
            result = list(set(self._members) - set(other._members))
            return Ensemble(members=result)
        return self  # Fallback

    def sym_difference(self, other):
        """Symmetric difference: (A ∖ B) ∪ (B ∖ A)."""
        if self._members is not None and other._members is not None:
            result = list(set(self._members) ^ set(other._members))
            return Ensemble(members=result)
        return self  # Fallback

    def realize(self, *, ordered=True):
        """Realize intensional definition to extensional members.
        
        In formal terms: evaluate the comprehension {x | P(x)} to get {x₁, x₂, ...}.
        """
        if self._members is not None:
            if ordered:
                return dedekind.ordered_set_roundtrip(self._members)
            else:
                return dedekind.unordered_set_roundtrip(self._members)
        raise NotImplementedError("Symbolic ensemble realization not yet implemented")

    def __repr__(self):
        if self._members is not None:
            return f"{{{', '.join(map(str, self._members))}}}"
        return f"Ensemble(intensional, P(x))"


print("expert-tier: shim types ready")

## Part 1: Intensional (Symbolic) Definitions

Define ensembles using set-builder comprehension and predicates. These exist symbolically until realized.

In [ ]:
# Intensional (symbolic, not yet computed)

# A = {1, 2, 3, 4, 5}
A = Ensemble.from_members(1, 2, 3, 4, 5)
print(f"A = {A}")

# B = {3, 4, 5, 6, 7}
B = Ensemble.from_members(3, 4, 5, 6, 7)
print(f"B = {B}")

# E = {n ∈ ℕ | 1 ≤ n ≤ 30, 2|n} (naturals from 1 to 30 divisible by 2)
E = Ensemble.comprehension(
    universe=range(1, 31),
    predicate=lambda n: n % 2 == 0
)
print(f"E = {{n ∈ [1..30] | n ≡ 0 (mod 2)}} (symbolic)")

# O = {n ∈ ℕ | 1 ≤ n ≤ 30, 2∤n} (odd naturals)
O = Ensemble.comprehension(
    universe=range(1, 31),
    predicate=lambda n: n % 2 == 1
)
print(f"O = {{n ∈ [1..30] | n ≡ 1 (mod 2)}} (symbolic)")

# Q = {n² | n ∈ E} (squares of evens)
Q = E.map_to(lambda n: n ** 2)
print(f"Q = {{n² | n ∈ E}} (symbolic)")

## Part 2: Extensional (Realized) Computation

Realize the intensional definitions to obtain concrete members.

In [ ]:
# Realize symbolic ensembles

# Set operations
A_cup_B = A.union(B).realize()       # A ∪ B
A_cap_B = A.intersection(B).realize() # A ∩ B
A_diff_B = A.difference(B).realize()  # A ∖ B

print("=== Extensional Results ===")
print(f"A ∪ B = {A_cup_B}")
print(f"A ∩ B = {A_cap_B}")
print(f"A ∖ B = {A_diff_B}")

# Realize comprehensions
evens_1_30 = E.realize()
odds_1_30 = O.realize()

print(f"E = {{n ∈ [1..30] | 2|n}} = {evens_1_30}")
print(f"O = {{n ∈ [1..30] | 2∤n}} = {odds_1_30}")

# Realize morphism image
squares = Q.realize()
print(f"Q = {{n² | n ∈ E}} = {squares}")

## Validation

Verify formal properties of the computed ensembles.

In [ ]:
# Verify set identities
assert A_cup_B == [1, 2, 3, 4, 5, 6, 7], f"A ∪ B failed: {A_cup_B}"
assert A_cap_B == [3, 4, 5], f"A ∩ B failed: {A_cap_B}"
assert A_diff_B == [1, 2], f"A ∖ B failed: {A_diff_B}"

# Verify comprehensions
assert evens_1_30 == [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30], f"Evens failed: {evens_1_30}"
assert odds_1_30 == [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29], f"Odds failed: {odds_1_30}"

# Verify morphism image preserves structure
assert len(squares) == len(evens_1_30), f"Image cardinality mismatch: {squares}"
assert squares[0] == 4, f"First square should be 2² = 4, got {squares[0]}"
assert squares[-1] == 900, f"Last square should be 30² = 900, got {squares[-1]}"

print("✓ All formal properties verified")
print("notebook-04-ok")